In [1]:
### Project Overview
# This projects seeks to answer the research question...
# Can text analysis of Indeed job listing descriptions identify recurring and role-specific skills and phrases useful 
# for resume tailoring?

In [2]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [3]:
### Experiment 1:
# Evaluate descriptions 1 for keywords using BOW
# Evaluate TF-IDF for descriptions 1 breaking documents by sentence

dataset = load_dataset("burcuonel/datajob_indeed_usa")

df = dataset["train"].to_pandas()

df['Descriptions'].head()

0    We invite you to join us at Driven Brands!\nHe...
1    Sabot Consulting (Sabot) is a management consu...
2    Job Details\nApply\nSave\nPrint this job\nEmai...
3    Position: Data Engineer\nLocation: MN\nAs a Da...
4    Full Job Description\nThe Network Administrato...
Name: Descriptions, dtype: str

In [4]:
# Explore description 1 for structure
description = df["Descriptions"].iloc[0]

# Split expression segments based on whitespace following characters (. , ! ?)
segments = re.split(r"(?<=[.!?])\s+|\n+", description)

segment = [segment.strip() for segment in segments if segment.strip()]

for index, segment in enumerate(segments):
    print(f'{index}: {segment}\n')

0: We invite you to join us at Driven Brands!

1: Headquartered in Charlotte, NC, Driven Brands (NASDAQ: DRVN) is the largest automotive services company in North America, providing a range of consumer and commercial automotive needs, including paint, collision, glass, vehicle repair, oil change, maintenance and car wash.

2: With over 4,500 centers in 15 countries, Driven Brands is the parent company of some of North America’s leading automotive service brands including Take 5 Oil Change, Take 5 Car Wash, Driven Glass, Meineke, Maaco, CARSTAR, and more.

3: Our network services over 50 million vehicles annually and generates more than $5 billion in system-wide sales each year.

4: Our culture inspires high performance and innovation, enabling our employees to go further, faster in their careers.

5: With amazing people and great brands, we confidently look forward to exciting growth ahead, and believe in following the values that support this vision.

6: JOB DESCRIPTION:

7: This is a

In [5]:
# Create unigram BOW for description 1
bow_vectorizer = CountVectorizer(
    analyzer="word", ngram_range=(1, 1), stop_words="english"
)

bow_matrix = bow_vectorizer.fit_transform([description])

print(bow_matrix.shape)

# Create labeled table of BOW counts, gathering results for top 15 unigrams
bow_results = pd.DataFrame(
    {"token": bow_vectorizer.get_feature_names_out(), "count": bow_matrix.toarray()[0]}
)

top_n = 15

bow_results.sort_values("count", ascending=False).head(top_n)

(1, 188)


,token,count
23,brands,6
48,data,6
72,experience,5
63,driven,4
38,company,3
49,datasets,3
17,automotive,3
8,analysis,3
115,models,3
74,field,3


In [6]:
# Create BOW for description 1 based on unigram
bow_vectorizer = CountVectorizer(
    analyzer="word", ngram_range=(2, 2), stop_words="english"
)

bow_matrix = bow_vectorizer.fit_transform([description])

print(bow_matrix.shape)

# Create labeled table of BOW counts, gathering results for top 15 unigrams
bow_results = pd.DataFrame(
    {"token": bow_vectorizer.get_feature_names_out(), "count": bow_matrix.toarray()[0]}
)

bow_results.sort_values("count", ascending=False).head(top_n)

(1, 231)


,token,count
84,driven brands,3
94,experience data,2
63,data analysis,2
53,company north,2
74,degree quantitative,2
41,car wash,2
153,north america,2
155,oil change,2
177,quantitative field,2
7,amazing people,1


In [7]:
# Apply TF-IDF across description 1 segments
tfidf_vectorizer = TfidfVectorizer(stop_words="english")

tfidf_matrix = tfidf_vectorizer.fit_transform(segments)

print(tfidf_matrix.shape)

# Create labeled table of TF-IDF tokens
tfidf_table = pd.DataFrame(
    tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out()
)

# Compare each segment with its highest weighted tokens
for index, segment in enumerate(segments):
    scores = tfidf_table.iloc[index]

    top_tokens = (
        scores[scores > 0]
        .sort_values(ascending=False)
        .head(5)
    )

    print(f'Segment {index}: {segment}')
    print(top_tokens)
    print()

(28, 188)
Segment 0: We invite you to join us at Driven Brands!
join      0.563466
invite    0.563466
driven    0.457165
brands    0.394983
Name: 0, dtype: float64

Segment 1: Headquartered in Charlotte, NC, Driven Brands (NASDAQ: DRVN) is the largest automotive services company in North America, providing a range of consumer and commercial automotive needs, including paint, collision, glass, vehicle repair, oil change, maintenance and car wash.
automotive    0.336925
consumer      0.189360
commercial    0.189360
collision     0.189360
charlotte     0.189360
Name: 1, dtype: float64

Segment 2: With over 4,500 centers in 15 countries, Driven Brands is the parent company of some of North America’s leading automotive service brands including Take 5 Oil Change, Take 5 Car Wash, Driven Glass, Meineke, Maaco, CARSTAR, and more.
driven       0.343008
brands       0.296353
500          0.211382
countries    0.211382
carstar      0.211382
Name: 2, dtype: float64

Segment 3: Our network services

In [ ]:
### Experiment 1 results
# The BOW model for description 1 yielded very little progress in favor of extracting useful keywords from a singular job description
# Unigrams appeared to contain less meaningful language than bigrams with english stop words, results without stop words were unusable
# TF-IDF at the sentence level also failed to isolate meaningful keywords that could be interpreted as leading terms

### Interpretation 
# These results provide a useful baseline for further study, but fail to prove that this singular job description surfaces ats-target keywords 
# based solely on frequency, or that a meaningful association exists between a keyword and description language

In [9]:
### Experiment 2
# Evaluate BOW model for keywords, short phrases across all descriptions
# Assess TF-IDF across all descriptions

descriptions = df['Descriptions'].dropna()
descriptions = descriptions[descriptions.str.strip().ne("")]

print(f'Documents: {len(descriptions)}')

Documents: 1200


In [10]:
# Create unigram BOW using all descriptions
unigram_vectorizer = CountVectorizer(
    analyzer="word", ngram_range=(1, 1), stop_words="english"
)

unigram_matrix = unigram_vectorizer.fit_transform(descriptions)

print(unigram_matrix.shape)

# Create counts for unigrams found in the documents
total_counts = unigram_matrix.sum(axis=0).A1
document_counts = (unigram_matrix > 0).sum(axis=0).A1

# Develop table for unigram stats
unigram_results = pd.DataFrame(
    {
        "token": unigram_vectorizer.get_feature_names_out(),
        "total_count": total_counts,
        "document_count": document_counts,
    }
)

# Find percent of documents the unigrams appear in
unigram_results["document_pct"] = (
    unigram_results["document_count"] / len(descriptions) * 100
)

top_n = 20

unigram_results.sort_values("total_count", ascending=False).head(top_n)

(1200, 14731)


,token,total_count,document_count,document_pct
4117,data,10183,1078,89.833333
5594,experience,6821,1140,95.000000
2649,business,4251,902,75.166667
14579,work,3757,1105,92.083333
13252,team,3125,914,76.166667
14682,years,2301,930,77.500000
12009,science,2219,743,61.916667
12425,skills,2156,900,75.000000
8082,learning,2035,589,49.083333
4485,development,1871,769,64.083333


In [15]:
# Explore bigram and trigram BOW using all descriptions without stop words
bigram_vectorizer = CountVectorizer(
    analyzer="word", ngram_range=(2, 2), stop_words="english"
)

bigram_matrix = bigram_vectorizer.fit_transform(descriptions)

print(bigram_matrix.shape)

# Create counts for bigrams found in the documents
total_counts = bigram_matrix.sum(axis=0).A1
document_counts = (bigram_matrix > 0).sum(axis=0).A1

# Develop table for bigram stats
bigram_results = pd.DataFrame(
    {
        "token": bigram_vectorizer.get_feature_names_out(),
        "total_count": total_counts,
        "document_count": document_counts,
    }
)

# Find percent of documents the unigrams appear in
bigram_results["document_pct"] = (
    bigram_results["document_count"] / len(descriptions) * 100
)

top_n = 30

bigram_results.sort_values("total_count", ascending=False).head(top_n)

(1200, 185462)


,token,total_count,document_count,document_pct
98690,machine learning,1461,480,40.000000
42165,data science,1414,453,37.750000
185051,years experience,755,451,37.583333
34120,computer science,556,475,39.583333
22656,business intelligence,449,207,17.250000
41311,data analytics,442,233,19.416667
17430,bachelor degree,429,385,32.083333
65032,experience working,426,297,24.750000
60993,equal opportunity,406,363,30.250000
182613,work location,405,398,33.166667


In [28]:
# Apply TF-IDF across Descriptions feature assessing bigrams
tfidf_vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(2, 2))

tfidf_matrix = tfidf_vectorizer.fit_transform(descriptions)
print(tfidf_matrix.shape)

# Assess bigram appearance in a single description
feature_names = tfidf_vectorizer.get_feature_names_out()

first_description_scores = pd.DataFrame(
    {"bigram": feature_names, "tfidf": tfidf_matrix[0].toarray().ravel()}
)

first_description_scores.query("tfidf > 0").sort_values('tfidf').head(top_n)

(1200, 185462)


,bigram,tfidf
34120,computer science,0.018476
42165,data science,0.018930
185051,years experience,0.018972
17430,bachelor degree,0.020487
139033,related field,0.024096
89957,job description,0.024927
42171,data scientists,0.025248
42204,data sources,0.029429
43959,decision making,0.029816
42357,data visualization,0.029948
